# Thinking pilot — inference sanity check

Loads the trained T1 LoRA adapter (`khairi/eshmun-thinking-pilot-lora`) on top of the base
checkpoint (`khairi/Eshmun-Thinking-Pilot`) and runs it on a few real pilot examples, to check
that the whole pipeline actually produces sensible, well-formatted output end to end.

**This is a qualitative sanity check, not the pilot evaluation.** The `-lora` adapter this
notebook loads was trained on the full undivided pilot dataset (2200 rows, no held-out split)
-- it predates `docs/thinking_pilot_evaluation_protocol.md`'s train/validation/test split
(`annotation_thinking`/`generation_thinking` configs on `khairi/eshmun-thinking-pilot`). Every
example below, including the ones pulled from the new `test` split, was almost certainly seen
during this model's training. Treat what you see here as "does the pipeline work, does the
model emit `<think>` blocks and valid sequences" -- not "how well does it generalize." The
real comparison (T1 vs. B1 direct-SFT ablation, on properly held-out data) needs both
conditions retrained against the split configs first (protocol §3.3, not done yet).

## 1. Imports + config

In [ ]:
import torch
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "khairi/Eshmun-Thinking-Pilot"
ADAPTER = "khairi/eshmun-thinking-pilot-lora"
DATASET = "khairi/eshmun-thinking-pilot"

DTYPE = torch.float32
MAX_NEW_TOKENS_ANNOTATION = 200
MAX_NEW_TOKENS_GENERATION = 600  # sequences can run long; think block + <protein>...</protein>
NUM_EXAMPLES_PER_DIRECTION = 3
SEED = 42

## 2. Load base model + adapter

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER)

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE)
model = PeftModel.from_pretrained(base_model, ADAPTER)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f"loaded {ADAPTER} on {BASE_MODEL}, device={device}")

## 3. Pull sample prompts from the pilot's test split

(See the caveat above -- "test" here means held out from the *new* split, not from what this specific adapter was trained on.)

In [ ]:
annotation_examples = load_dataset(DATASET, "annotation_thinking", split="test").shuffle(seed=SEED).select(range(NUM_EXAMPLES_PER_DIRECTION))
generation_examples = load_dataset(DATASET, "generation_thinking", split="test").shuffle(seed=SEED).select(range(NUM_EXAMPLES_PER_DIRECTION))

print("annotation examples:", len(annotation_examples))
print("generation examples:", len(generation_examples))

## 4. Generation helper + format checks

In [ ]:
import re

THINK_RE = re.compile(r"^<think>\n.*?\n</think>\n", re.DOTALL)
VALID_AA = frozenset("ACDEFGHIKLMNPQRSTVWY")
PROTEIN_RE = re.compile(r"<protein>(.*?)</protein>", re.DOTALL)


def generate(prompt: str, max_new_tokens: int) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # completion is whatever follows the prompt
    return full_text[len(tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)):]


def check_think_format(completion: str) -> bool:
    return bool(THINK_RE.match(completion))


def check_sequence_validity(completion: str) -> bool | None:
    match = PROTEIN_RE.search(completion)
    if not match:
        return None  # no <protein> tag found at all
    seq = match.group(1).replace("Ƥ", "")
    return bool(seq) and all(c in VALID_AA for c in seq.upper())

## 5. Annotation direction (sequence → property)

In [ ]:
for i, example in enumerate(annotation_examples):
    print(f"=== annotation example {i} (entry={example['Entry']}) ===")
    completion = generate(example["Instruction"], MAX_NEW_TOKENS_ANNOTATION)
    print("generated:", completion[:500])
    print("true answer:", example["Answer"])
    print("well-formed <think> block:", check_think_format(completion))
    print()

## 6. Generation direction (property → sequence)

In [ ]:
for i, example in enumerate(generation_examples):
    print(f"=== generation example {i} (entry={example['Entry']}) ===")
    print("instruction:", example["Instruction"])
    completion = generate(example["Instruction"], MAX_NEW_TOKENS_GENERATION)
    print("generated (truncated):", completion[:300])
    print("well-formed <think> block:", check_think_format(completion))
    print("sequence validity (None = no <protein> tag found):", check_sequence_validity(completion))
    print()

## 7. Next steps

If this looks broken (no `<think>` tags, garbage/invalid sequences, empty completions),
something is wrong with the checkpoint or the loading code here -- fix that before trusting
anything from the actual evaluation.

If this looks reasonable, the next real step is protocol §3.3: retrain both T1 and B1 against
the `*_thinking`/`*_non_thinking` split configs (not the full undivided dataset this adapter
used), then build the scoring script from §9 checklist items 3/4/6 to get the actual go/no-go
numbers -- this notebook only checks that generation runs and looks superficially sane, it
doesn't compute any of the protocol's actual metrics.